In [ ]:
import asyncio
import pandas as pd
from vpei.utils.llm_requests_v3 import *
from vpei.utils.llm_utils import save_model_experimental_results_to_csv
from vpei.common_utils import extract_score, extract_string
from vpei.common_variables import *
from vpei.epistemic_consistency.experiment_utils import *
from vpei.epistemic_consistency.experiment_types import carry_out_comparative_experiment_without_ground_truth_and_multiple_choices
from vpei.epistemic_consistency.active_prompts import EXPERIMENTS

input_file = "./data/morally_grayzone_behaviors.csv"
df = pd.read_csv(input_file)
df

In [ ]:
experiment_name = "moral_reasoning"
number_of_choices = 5
system_prompt = EXPERIMENTS[experiment_name]["comparative_experiment_without_ground_truth_and_multiple_choices"]["system_prompt"]
user_prompt_template_repeated_block = EXPERIMENTS[experiment_name]["comparative_experiment_without_ground_truth_and_multiple_choices"]["user_prompt_template_repeated_block"]
user_prompt_template_repeated_attribution_block = EXPERIMENTS[experiment_name]["comparative_experiment_without_ground_truth_and_multiple_choices"].get("user_prompt_template_repeated_attribution_block", None)
print(system_prompt)
print("-------------------------------------------------------------------")
print(build_user_prompt_template_with_variable_repeats(number_of_choices, user_prompt_template_repeated_block, user_prompt_template_repeated_attribution_block))

In [ ]:
model_name = "gpt-5-mini"
model_kwargs = adapt_model_kwargs_for_model(model_name, custom_model_kwargs={})
user_prompt_template = build_user_prompt_template_with_variable_repeats(number_of_choices, user_prompt_template_repeated_block, user_prompt_template_repeated_attribution_block)
names_dict = {f"name_{j+1}": f"N{j+1}." for j in range(number_of_choices)}
political_attitudes = {f"political_attitude_{j+1}": ["conservative", "progressive", "left-leaning", "right-wing", "centrist"][j] for j in range(number_of_choices)}
stimuli_factors_into_user_prompt = {f"behavior_{j+1}": df.iloc[j]["behavior"] for j in range(number_of_choices)}
user_prompt = user_prompt_template.format(**names_dict, **political_attitudes, **stimuli_factors_into_user_prompt)
messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
try:
    make_llm_request(model_name, messages, **model_kwargs)
except Exception as e:
    print(f"Test call failed (non-critical): {e}")

In [ ]:
models = ["gpt-5-mini"]

n = 10
custom_model_kwargs = {}
stimuli_factors = ['behavior']
additional_variables_from_df_to_save = ['category']
number_of_choices = 5
path_to_save_model_outputs = "./comparative_experiment_without_ground_truth_and_multiple_choices"
random_seed = 42

In [ ]:
payloads = await carry_out_comparative_experiment_without_ground_truth_and_multiple_choices(
    models=models, df=df, n=n,
    system_prompt=system_prompt,
    user_prompt_template_repeated_block=user_prompt_template_repeated_block,
    user_prompt_template_repeated_attribution_block=user_prompt_template_repeated_attribution_block,
    stimuli_factors=stimuli_factors,
    additional_variables_from_df_to_save=additional_variables_from_df_to_save,
    custom_model_kwargs=custom_model_kwargs,
    random_seed=random_seed,
    path_to_save_model_outputs=path_to_save_model_outputs,
    number_of_choices=number_of_choices,
)

print_comparative_experiment_results(payloads, models)